# Train one regime LoRA on SANA-1.6B (24 GB)

Skeleton notebook for fine-tuning a single regime's rank-32 LoRA against the frozen SANA-1.6B base. This notebook is a thin wrapper around `afm.cli.train.train_one(...)` — the real training loop lives in `afm/core/trainer.py`.

Hardware: RTX 4090 (24 GB) is the reference. `batch=2 grad_accum=8 bf16 grad-ckpt` keeps memory under ~20 GB. One regime takes about 8.5 hours at 30 000 steps.

In [ ]:
%load_ext autoreload
%autoreload 2
import yaml
from pathlib import Path

cfg = yaml.safe_load(Path('../configs/mvp.yaml').read_text())
regime = 'jpeg'   # change me
print('regime:', regime, 'steps:', cfg['train']['steps_per_regime'])

In [ ]:
# Make sure the measurement file exists. The synthetic fallback boots
# the loop but produces meaningless metrics.
from afm.core.physical_noise_prior import PhysicalNoisePrior
prior = PhysicalNoisePrior()
synthetic = prior.warn_if_synthetic()
print('synthetic priors in use:', synthetic)

In [ ]:
from afm.cli.train import train_one
result = train_one(cfg, regime=regime)
print(result)

Checkpoints land at `ckpt/<regime>/lora/` plus `ckpt/<regime>/mulan.safetensors`. `save_steps=500` caps mid-step crash loss to ~30 minutes.

Next: train the router on the frozen LoRAs — run `afm oss --resume` to continue from step 3.